In [2]:
import pandas as pd, numpy as np, json, os

os.makedirs("../docs", exist_ok=True)

raw = pd.read_csv("../data/raw/USvideos.csv")
clean = pd.read_csv("../data/clean/posts_clean.csv")
with open("../data/raw/US_category_id.json") as f:
    cats = json.load(f)
cat_map = {int(i["id"]): i["snippet"]["title"] for i in cats["items"]}
raw["category"] = raw["category_id"].map(cat_map)

for c in ["is_deleted", "is_zero_views", "is_outlier"]:
    clean[c] = clean[c].astype(bool)

raw_rows = len(raw)
raw_unique = raw["video_id"].nunique()
cat_rows = raw["category"].isin(["Sports", "Entertainment"]).sum()
clean_rows = len(clean)
n_deleted = int(clean["is_deleted"].sum())
n_zero = int(clean["is_zero_views"].sum())
n_out = int(clean["is_outlier"].sum())
q1, q3 = clean["views"].quantile([0.25, 0.75])
cutoff = int(q3 + 1.5 * (q3 - q1))
other_share = round((clean["content_type"] == "Other").mean() * 100, 1)

valid = clean[(~clean["is_zero_views"]) & (~clean["is_deleted"])]
avg_er = round(valid["engagement_rate"].mean(), 3)
med_er = round(valid["engagement_rate"].median(), 3)
avg_er_no_out = round(valid[~valid["is_outlier"]]["engagement_rate"].mean(), 3)

by_day = valid.groupby("post_day")["engagement_rate"].agg(videos="count", avg_er="mean").round(3).sort_values("avg_er", ascending=False)
by_hour = valid.groupby("post_hour")["engagement_rate"].agg(videos="count", avg_er="mean").round(3)
by_hour = by_hour[by_hour["videos"] >= 30].sort_values("avg_er", ascending=False)
by_type = valid.groupby("content_type")["engagement_rate"].agg(videos="count", avg_er="mean").round(3).sort_values("avg_er", ascending=False)
by_cat = valid.groupby("category")["engagement_rate"].agg(videos="count", avg_er="mean").round(3).sort_values("avg_er", ascending=False)

def md(df, idx_name):
    df = df.reset_index().rename(columns={df.index.name or "index": idx_name})
    head = "| " + " | ".join(df.columns) + " |"
    sep = "|" + "---|" * len(df.columns)
    rows = ["| " + " | ".join(str(v) for v in r) + " |" for r in df.values]
    return "\n".join([head, sep] + rows)

def write(path, text):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text.strip() + "\n")

best_day, best_day_er = by_day.index[0], by_day.iloc[0]["avg_er"]
worst_day, worst_day_er = by_day.index[-1], by_day.iloc[-1]["avg_er"]
best_hour, best_hour_er = int(by_hour.index[0]), by_hour.iloc[0]["avg_er"]
top_type, top_type_er = by_type.index[0], by_type.iloc[0]["avg_er"]
low_type, low_type_er = by_type.index[-1], by_type.iloc[-1]["avg_er"]
top_cat, top_cat_er = by_cat.index[0], by_cat.iloc[0]["avg_er"]
low_cat, low_cat_er = by_cat.index[-1], by_cat.iloc[-1]["avg_er"]

write("../docs/problem_statement.md", f"""
# Problem Statement and KPI Definitions

## Project
Project 24: Social Media Engagement Analysis (Sports and Entertainment)

## Problem
Content creators and media teams want to know which kinds of videos, and which posting
days and hours, earn the most audience engagement. This project analyses YouTube trending
videos in the Sports and Entertainment categories to find how engagement rate is defined and
calculated, the best day and hour to post, and the most engaging content types.

## Dataset
Kaggle "Trending YouTube Video Statistics" (US videos): USvideos.csv and US_category_id.json.
Filtered to the Sports and Entertainment categories.

## Tools
Python (pandas) for cleaning, SQL for engagement-rate queries (run in SQLite, Oracle-syntax
scripts provided), Power BI for the dashboard.

## KPI definitions
| KPI | Definition |
|---|---|
| Engagement Rate (ER) | (likes + comments) / views x 100 |
| Average ER | Mean ER over valid videos |
| Median ER | Middle ER value over valid videos |
| Total Videos | Count of valid videos |
| Total Views | Sum of views over valid videos |
| Best posting day | Weekday with the highest average ER (IST) |
| Best posting hour | Hour with the highest average ER (IST, hours with at least 30 videos) |
| Viral outlier | Video with views above Q3 + 1.5 x IQR |

## Data rules
- Zero-view videos: flagged, ER left blank, excluded from averages.
- Deleted or removed videos: flagged and excluded from analysis.
- Viral outliers: flagged but kept, with a dashboard toggle.
- One row per video: the row with the highest views is kept.
""")

write("../docs/data_quality_report.md", f"""
# Data Quality Report

## Row counts
| Stage | Rows |
|---|---|
| Raw file (USvideos.csv) | {raw_rows:,} |
| Unique videos in raw file | {raw_unique:,} |
| Rows in Sports and Entertainment (before removing repeats) | {cat_rows:,} |
| Rows after cleaning (one row per video) | {clean_rows:,} |

The raw file lists the same video on many trending days, so repeated rows were removed and
the row with the highest views was kept for each video.

## Issues found and how they were handled
| Issue | Count | Handling |
|---|---|---|
| Deleted or removed videos | {n_deleted:,} | Flagged (is_deleted), excluded from analysis |
| Zero-view videos | {n_zero:,} | Flagged (is_zero_views), ER left blank, excluded from averages |
| Viral outliers | {n_out:,} | Flagged (is_outlier), kept; cutoff = {cutoff:,} views (Q3 + 1.5 x IQR) |
| Valid videos used for ER | {len(valid):,} | |

Non-numeric views, likes and comments were converted to numbers, with blanks set to 0.

## Derived fields
- engagement_rate = (likes + comment_count) / views x 100
- post_day and post_hour from publish_time converted from UTC to IST (Asia/Kolkata)
- category from US_category_id.json
- content_type assigned by keyword rules on title and channel name

## Known limitation
{other_share}% of videos did not match any content-type keyword rule and are labelled "Other".

## Verification
Engagement rate was verified for 10 random videos against the raw file and by hand
(see verification.md). Edge cases are in test_cases.md.
""")

write("../docs/insights.md", f"""
# Insights Summary

All times are IST. Figures use valid videos only ({len(valid):,} videos).

## Key findings
1. Overall engagement: average ER is {avg_er}% and median ER is {med_er}%.
2. Best day to post: {best_day} has the highest average ER ({best_day_er}%); {worst_day} has the lowest ({worst_day_er}%).
3. Best hour to post: {best_hour}:00 IST has the highest average ER ({best_hour_er}%).
4. Content types: {top_type} ranks first ({top_type_er}%) and {low_type} ranks last ({low_type_er}%).
5. Category comparison: {top_cat} ({top_cat_er}%) vs {low_cat} ({low_cat_er}%).
6. Effect of viral videos: average ER is {avg_er}% with outliers and {avg_er_no_out}% without them.

## Tables
### ER by day
{md(by_day, "post_day")}

### ER by hour (top 10, hours with at least 30 videos)
{md(by_hour.head(10), "post_hour")}

### ER by content type
{md(by_type, "content_type")}

### ER by category
{md(by_cat, "category")}

## Recommendation
Post {top_type} content on {best_day}s around {best_hour}:00 IST to target the highest engagement.

## Limitations
- The data is YouTube trending videos, not all uploads, so results describe trending content.
- Content types come from keyword rules on titles, and {other_share}% are "Other".
- Engagement rate is a ratio, so videos with very few views can have unusually high rates.
""")

print("Docs written")
print("Best day:", best_day, "| Best hour:", best_hour, "| Top type:", top_type)

Docs written
Best day: Saturday | Best hour: 1 | Top type: Behind the Scenes / Vlog


In [3]:
write("../README.md", f"""
# Social Media Engagement Analysis (Project 24)

Analysis of YouTube trending videos in Sports and Entertainment to find which content types,
days and hours get the highest engagement.

## Summary of results
- Valid videos analysed: {len(valid):,}
- Average ER: {avg_er}% | Median ER: {med_er}%
- Best day: {best_day} | Best hour: {best_hour}:00 IST
- Top content type: {top_type}

## Dataset
Kaggle: Trending YouTube Video Statistics (USvideos.csv, US_category_id.json).

## Tools
Python (pandas), SQLite (run in VS Code), Oracle-syntax SQL scripts, Power BI Desktop, Git/GitHub.

## Engagement rate formula
ER = (likes + comments) / views x 100

## Folder structure
- data/raw/ : original files
- data/clean/ : cleaned CSV, SQLite database, SQL result CSVs
- notebooks/ : 01_cleaning, 02_sql, 03_verification, 04_docs
- sql/ : SQLite scripts (ddl, load, views, queries)
- sql/oracle/ : Oracle-syntax versions of the same scripts
- powerbi/ : engagement_dashboard.pbix
- docs/ : problem statement, data quality report, insights, tests, diagram

## How to run
1. Run notebooks/01_cleaning.ipynb to create data/clean/posts_clean.csv.
2. Run notebooks/02_sql.ipynb to create the database, views and query results.
3. Run notebooks/03_verification.ipynb to verify 10 videos.
4. Open powerbi/engagement_dashboard.pbix in Power BI Desktop.

## Note on Oracle
Oracle XE installation failed on my machine, so the queries were executed in SQLite using
standard SQL. Oracle-syntax scripts (DDL, load, views, queries) are provided in sql/oracle/.

## Demo video
(paste your YouTube link here)

## Author
Varnita Shrivastava, B.Tech CSE (Data Science), ABES Engineering College
""")

print("README written")

README written
